# Few-Shot Learning with Random Forest on Titanic

Few-shot learning trains models with very limited labeled data. In this notebook you will:
- Understand the challenge of limited training data
- Train a RandomForestClassifier with 5, 10, 20, 50 examples (few-shot)
- Compare against full-dataset training (standard learning)
- Analyze how performance scales with training examples

## Problem Definition

Few-shot learning addresses three core challenges:
1. **Data scarcity**: labeled data is expensive to collect
2. **Quick adaptation**: models must generalize from minimal examples
3. **Resource constraints**: limited time and compute for data collection

For Titanic survival prediction, imagine you only have records for a few passengers.

In [ ]:
import numpy as np, pandas as pd, seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings; warnings.filterwarnings('ignore')

## Prepare Dataset

In [ ]:
def load_titanic():
    df = sns.load_dataset('titanic')
    df.drop(['class','who','adult_male','deck','embark_town','alive','alone'], axis=1, inplace=True)
    df['age'].fillna(df['age'].median(), inplace=True)
    df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)
    df['sex'] = df['sex'].map({'male':1,'female':0})
    df['embarked'] = LabelEncoder().fit_transform(df['embarked'])
    return df

df = load_titanic()
X = df.drop('survived', axis=1)
y = df['survived']
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler().fit(X_train_full)
X_train_full_s = scaler.transform(X_train_full)
X_test_s       = scaler.transform(X_test)
print(f"Full training set: {X_train_full_s.shape[0]} examples")
print(f"Test set:          {X_test_s.shape[0]} examples")

## Full Dataset Baseline (Standard Learning)

In [ ]:
# Train with all available data — this is the upper bound
full_model = RandomForestClassifier(n_estimators=100, random_state=42)
full_model.fit(X_train_full_s, y_train_full)
full_acc = accuracy_score(y_test, full_model.predict(X_test_s))
full_f1  = f1_score(y_test, full_model.predict(X_test_s))
print(f"Full dataset ({X_train_full_s.shape[0]} examples):")
print(f"  Accuracy: {full_acc:.4f}, F1: {full_f1:.4f}")

## Few-Shot Experiments

Train with only 5, 10, 20, 50, 100 examples per experiment and evaluate performance.

In [ ]:
# Define few-shot example counts (analogous to few-shot 'k' values)
few_shot_sizes = [5, 10, 20, 50, 100]

# Example database — mirroring the CustomerServiceBot's example_database structure
example_database = {
    'very_few':     5,
    'few':         10,
    'moderate':    20,
    'small':       50,
    'medium':     100,
}

def load_examples(category):
    '''Retrieve few-shot examples for a given category.'''
    n = example_database[category]
    idx = np.random.choice(len(X_train_full_s), size=n, replace=False)
    return X_train_full_s[idx], y_train_full.iloc[idx]

results = {}

for size in few_shot_sizes:
    # Sample k examples for few-shot training
    idx = np.random.choice(len(X_train_full_s), size=size, replace=False)
    X_few = X_train_full_s[idx]
    y_few = y_train_full.iloc[idx]

    # Train with few examples
    few_model = RandomForestClassifier(n_estimators=min(size, 50), random_state=42)
    few_model.fit(X_few, y_few)

    acc = accuracy_score(y_test, few_model.predict(X_test_s))
    f1  = f1_score(y_test, few_model.predict(X_test_s), zero_division=0)
    results[size] = {'accuracy': acc, 'f1': f1}
    print(f"{size:4d} examples: Accuracy={acc:.4f}, F1={f1:.4f}")

print(f"\nFull dataset ({len(X_train_full_s)} examples): Accuracy={full_acc:.4f}, F1={full_f1:.4f}")

## Analyze Results

In [ ]:
sizes = list(results.keys()) + [len(X_train_full_s)]
accs  = [results[s]['accuracy'] for s in list(results.keys())] + [full_acc]
f1s   = [results[s]['f1'] for s in list(results.keys())] + [full_f1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.semilogx(sizes, accs, 'o-', color='steelblue', linewidth=2, markersize=8)
ax1.axhline(y=full_acc, color='red', linestyle='--', label=f'Full data ({full_acc:.3f})')
ax1.set_xlabel('Number of Training Examples (log scale)')
ax1.set_ylabel('Accuracy')
ax1.set_title('Few-Shot Learning: Accuracy vs Training Size')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.semilogx(sizes, f1s, 's-', color='coral', linewidth=2, markersize=8)
ax2.axhline(y=full_f1, color='blue', linestyle='--', label=f'Full data ({full_f1:.3f})')
ax2.set_xlabel('Number of Training Examples (log scale)')
ax2.set_ylabel('F1 Score')
ax2.set_title('Few-Shot Learning: F1 Score vs Training Size')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

The plot shows how performance scales with the number of training examples. With very few examples (5-10), the Random Forest struggles due to insufficient data. Performance improves rapidly as more examples are added, eventually approaching the full-data baseline.